# HH2 - Solar predictions vs light sensor values

### Import libraries 

In [92]:
#Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn' ## disable chained assignment warnings due to false positives
#NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. We will use this in our plotter function to plot data.
import matplotlib.pyplot as plt
#Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns
#Plotly is a Python data visualization library 
import plotly.graph_objects as go
import plotly.express as px
import datetime as dt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Import metrics
from sklearn.ensemble import RandomForestRegressor

### Load and view data 

In [94]:
# Get the merged database (saved at the end of notebook STEP 1)
database_hourly =  pd.read_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\Year4_CoolAI\Jupyter_notebooks\H2\DATA (backups)\1_hourly.csv")
database_hourly.drop(['weather_main', "Temperature_outdoor"], axis=1, inplace=True)
database_hourly['ts'] = pd.to_datetime(database_hourly['ts'])  # Specify the correct format

database_hourly = database_hourly[["ts", "light_left", "light_right"]]
database_hourly.tail()

,ts,light_left,light_right
661,2024-09-25 14:00:00,3259.0,2686.0
662,2024-09-25 15:00:00,3665.0,2871.0
663,2024-09-25 16:00:00,3315.0,3600.0
664,2024-09-25 17:00:00,2630.0,2813.0
665,2024-09-25 18:00:00,1442.0,1578.0


In [144]:
df_SolarAPI = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/cytoZHE1NnlrQ20wNlJHRlU2MW1GNDlFTnhGNy8xcmpLSmJFVU8vRWYrbz0=")
df_SolarAPI = df_SolarAPI[["tijd_nl", 'gr']]
# df_SolarAPI['tijd_nl']
df_SolarAPI['ts'] = pd.to_datetime(df_SolarAPI['tijd_nl'], format="%d-%m-%Y %H:%M")  # Specify the correct format
df_SolarAPI = df_SolarAPI.drop(columns=['tijd_nl'])
df_SolarAPI.head()

,gr,ts
0,0.0,2024-09-10 21:00:00
1,0.0,2024-09-10 22:00:00
2,0.0,2024-09-10 23:00:00
3,0.0,2024-09-11 00:00:00
4,0.0,2024-09-11 01:00:00


In [148]:
### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_SolarAPI,
    database_hourly,
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_merge = reduce(merge_asof, dataframes_to_merge)

# Drop rows where both 'light_left' and 'light_right' columns have no values (NaN)
df_merge = df_merge.dropna(subset=['light_left', 'light_right'], how='all')
df_merge = df_merge.dropna()

# feature engineering: extract the hour from the ts column
df_merge['hr'] = df_merge['ts'].dt.hour

df_merge

,gr,ts,light_left,light_right,hr
0,5.0,2024-09-02 19:00:00,2225.0,1535.0,19
1,2.0,2024-09-02 20:00:00,992.0,655.0,20
2,0.0,2024-09-02 21:00:00,0.0,0.0,21
3,0.0,2024-09-02 22:00:00,0.0,0.0,22
4,0.0,2024-09-02 23:00:00,0.0,0.0,23
...,...,...,...,...,...
169,6.0,2024-09-25 18:00:00,1442.0,1578.0,18
170,1.0,2024-09-25 19:00:00,1442.0,1578.0,19
171,1.0,2024-09-25 19:00:00,1442.0,1578.0,19
172,0.0,2024-09-25 20:00:00,1442.0,1578.0,20


In [150]:
# Prepare the Data
X = df_merge[['gr', 'hr']]
y_left = df_merge['light_left']
y_right = df_merge['light_right']
   
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_left_train, y_left_test = train_test_split(X, y_left, test_size=0.2, random_state=42)
X_train, X_test, y_right_train, y_right_test = train_test_split(X, y_right, test_size=0.2, random_state=42)

# choose a model (linear regression, random forest regressor, or gradient boosting regressor)
from sklearn.ensemble import RandomForestRegressor
model_left = RandomForestRegressor()
model_right = RandomForestRegressor()

# from sklearn.linear_model import LinearRegression
# model_left = LinearRegression()
# model_right = LinearRegression()


# Train the model
model_left.fit(X_train, y_left_train)
model_right.fit(X_train, y_right_train)

# Make predictions
y_left_pred = model_left.predict(X_test)
y_right_pred = model_right.predict(X_test)

# Evaluate the model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Light Left - MAE:", mean_absolute_error(y_left_test, y_left_pred))
print("Light Right - MAE:", mean_absolute_error(y_right_test, y_right_pred))
print("Light Left - R^2:", r2_score(y_left_test, y_left_pred))
print("Light Right - R^2:", r2_score(y_right_test, y_right_pred))

Light Left - MAE: 315.0226916486291
Light Right - MAE: 271.7423250568876
Light Left - R^2: 0.9091269220324261
Light Right - R^2: 0.9256970565219044


# Save the trained model

In [152]:
import joblib

# Save the trained models to disk
joblib.dump(model_left, 'model_left.pkl')  # Save the left light model
joblib.dump(model_right, 'model_right.pkl')  # Save the right light model

['model_right.pkl']